<a href="https://colab.research.google.com/github/Harsh-2404/ecommerce-book-analytics-pipeline/blob/main/1_Web_Scraping_Books.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone Project - Phase 1: Web Scraping
**Dataset Source:** http://books.toscrape.com/

**Target Records:** 1,000 Books across 50 Pages

**Goal:** Extract book information for E-Commerce analytics and price prediction.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

In [ ]:
BASE_URL = "https://books.toscrape.com/"
TOTAL_PAGES = 50

print("Base URL:", BASE_URL)
print("Target pages:", TOTAL_PAGES)

Base URL: https://books.toscrape.com/
Target pages: 50


Test Website Connection

In [ ]:
response = requests.get(BASE_URL, timeout=30)

print("Status Code:", response.status_code)

if response.status_code == 200:
    print("Website connection successful.")
else:
    print("Website connection failed.")

Status Code: 200
Website connection successful.


Parse HTML

In [ ]:
soup = BeautifulSoup(response.text, "html.parser")

print("HTML parsing completed successfully.")

HTML parsing completed successfully.


Inspect Number of Books on First Page

In [ ]:
book_cards = soup.find_all("article", class_="product_pod")

print("Books found on the first page:", len(book_cards))

Books found on the first page: 20


Inspect One Book

In [ ]:
first_book = book_cards[0]

print(first_book.prettify()[:3000])

<article class="product_pod">
 <div class="image_container">
  <a href="catalogue/a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   Â£51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
</article>



Extract One Book

In [ ]:
title = first_book.h3.a["title"]

price_text = first_book.find("p", class_="price_color").get_text(strip=True)

rating_class = first_book.find("p", class_="star-rating")["class"]        # here ["class"] :# Extract classes as a list ['star-rating', 'Three'] to get the rating text from index 1
rating_text = rating_class[1]

availability = first_book.find(
    "p", class_="instock availability"
).get_text(" ", strip=True)

relative_url = first_book.h3.a["href"]

product_url = BASE_URL + relative_url

print("Sample Book Details")
print("--------------------")
print("Title:", title)
print("Price:", price_text)
print("Rating:", rating_text)
print("Availability:", availability)
print("Product URL:", product_url)

Sample Book Details
--------------------
Title: A Light in the Attic
Price: Â£51.77
Rating: Three
Availability: In stock
Product URL: https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html


Create Rating Mapping

In [ ]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

print("Rating mapping created:")
print(rating_map)

Rating mapping created:
{'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}


Create Scraping Function

In [ ]:
def scrape_book(book, base_url):
    """
    Extract relevant information from a single book card.
    """

    title = book.h3.a["title"]

    price_text = book.find(
        "p", class_="price_color"
    ).get_text(strip=True)

    price_gbp = float(
        price_text.replace("£", "").replace("Â", "").strip()
    )

    rating_text = book.find(
        "p", class_="star-rating"
    )["class"][1]

    rating_stars = rating_map.get(rating_text)

    stock_status = book.find(
        "p", class_="instock availability"
    ).get_text(" ", strip=True)

    relative_url = book.h3.a["href"]
    product_url = base_url + relative_url

    return {
        "Title": title,
        "Price_GBP": price_gbp,
        "Rating_Stars": rating_stars,
        "Stock_Status": stock_status,
        "Product_URL": product_url
    }

Test the Function

In [ ]:
sample_book = scrape_book(first_book, BASE_URL)

print("Sample extracted record:")
print(sample_book)

Sample extracted record:
{'Title': 'A Light in the Attic', 'Price_GBP': 51.77, 'Rating_Stars': 3, 'Stock_Status': 'In stock', 'Product_URL': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'}


Scrape All 50 Pages

In [ ]:
# Initialize an empty list to store all scraped book records
books_data = []

print("Starting web scraping...")
print(f"Target: {TOTAL_PAGES} pages")
print("-" * 50)

for page in range(1, TOTAL_PAGES + 1):

    # Construct the URL for the current page
    page_url = f"{BASE_URL}catalogue/page-{page}.html"

    try:
        # Send request to the webpage
        response = requests.get(page_url, timeout=30)

        # Check whether the request was successful
        if response.status_code != 200:
            print(f"Page {page}: Failed - Status Code {response.status_code}")
            continue

        # Parse the HTML content
        soup = BeautifulSoup(response.text, "html.parser")

        # Find all book cards on the current page
        book_cards = soup.find_all(
            "article",
            class_="product_pod"
        )

        # Extract information from each book
        for book in book_cards:
            book_record = scrape_book(book, BASE_URL)
            books_data.append(book_record)

        # Display progress after each page
        print(
            f"Page {page:02d}/{TOTAL_PAGES}: "
            f"{len(book_cards)} books scraped | "
            f"Total records: {len(books_data)}"
        )

        # Small delay between requests
        time.sleep(0.5)

    except requests.RequestException as error:
        print(f"Page {page}: Request failed - {error}")
        continue

print("-" * 50)
print("Web scraping completed.")
print(f"Total records collected: {len(books_data)}")

Starting web scraping...
Target: 50 pages
--------------------------------------------------
Page 01/50: 20 books scraped | Total records: 20
Page 02/50: 20 books scraped | Total records: 40
Page 03/50: 20 books scraped | Total records: 60
Page 04/50: 20 books scraped | Total records: 80
Page 05/50: 20 books scraped | Total records: 100
Page 06/50: 20 books scraped | Total records: 120
Page 07/50: 20 books scraped | Total records: 140
Page 08/50: 20 books scraped | Total records: 160
Page 09/50: 20 books scraped | Total records: 180
Page 10/50: 20 books scraped | Total records: 200
Page 11/50: 20 books scraped | Total records: 220
Page 12/50: 20 books scraped | Total records: 240
Page 13/50: 20 books scraped | Total records: 260
Page 14/50: 20 books scraped | Total records: 280
Page 15/50: 20 books scraped | Total records: 300
Page 16/50: 20 books scraped | Total records: 320
Page 17/50: 20 books scraped | Total records: 340
Page 18/50: 20 books scraped | Total records: 360
Page 19/50:

Create the Raw DataFrame

In [ ]:
# Convert the scraped records into a pandas DataFrame
df_raw = pd.DataFrame(books_data)

print("Raw DataFrame created successfully.")
print(f"Total records: {len(df_raw)}")
print(f"Total columns: {len(df_raw.columns)}")

Raw DataFrame created successfully.
Total records: 1000
Total columns: 5


Display the First 10 Records

In [ ]:
# Display the first 10 records
display(df_raw.head(10))

,Title,Price_GBP,Rating_Stars,Stock_Status,Product_URL
0,A Light in the Attic,51.77,3,In stock,https://books.toscrape.com/a-light-in-the-atti...
1,Tipping the Velvet,53.74,1,In stock,https://books.toscrape.com/tipping-the-velvet_...
2,Soumission,50.10,1,In stock,https://books.toscrape.com/soumission_998/inde...
3,Sharp Objects,47.82,4,In stock,https://books.toscrape.com/sharp-objects_997/i...
4,Sapiens: A Brief History of Humankind,54.23,5,In stock,https://books.toscrape.com/sapiens-a-brief-his...
5,The Requiem Red,22.65,1,In stock,https://books.toscrape.com/the-requiem-red_995...
6,The Dirty Little Secrets of Getting Your Dream...,33.34,4,In stock,https://books.toscrape.com/the-dirty-little-se...
7,The Coming Woman: A Novel Based on the Life of...,17.93,3,In stock,https://books.toscrape.com/the-coming-woman-a-...
8,The Boys in the Boat: Nine Americans and Their...,22.60,4,In stock,https://books.toscrape.com/the-boys-in-the-boa...
9,The Black Maria,52.15,1,In stock,https://books.toscrape.com/the-black-maria_991...


Check Column Names

In [ ]:
# Display all column names
print("Dataset Columns:")
print(df_raw.columns.tolist())

Dataset Columns:
['Title', 'Price_GBP', 'Rating_Stars', 'Stock_Status', 'Product_URL']


Check Dataset Dimensions

In [ ]:
# Check the number of rows and columns
rows, columns = df_raw.shape

print("Dataset Dimensions")
print("-------------------")
print(f"Rows: {rows}")
print(f"Columns: {columns}")

Dataset Dimensions
-------------------
Rows: 1000
Columns: 5


Check Data Types

In [ ]:
# Check the data types of all columns
print("Data Types")
print("----------")
print(df_raw.dtypes)

Data Types
----------
Title            object
Price_GBP       float64
Rating_Stars      int64
Stock_Status     object
Product_URL      object
dtype: object


Check Missing Values

In [ ]:
# Check for missing values in each column
missing_values = df_raw.isnull().sum()

print("Missing Values")
print("--------------")
print(missing_values)

Missing Values
--------------
Title           0
Price_GBP       0
Rating_Stars    0
Stock_Status    0
Product_URL     0
dtype: int64


Check Duplicate Records

In [ ]:
# Check for completely duplicated records
duplicate_count = df_raw.duplicated().sum()

print("Duplicate Records")
print("------------------")
print(f"Total duplicate rows: {duplicate_count}")

Duplicate Records
------------------
Total duplicate rows: 0


Save Raw Dataset

In [ ]:
# Save the raw scraped dataset as a CSV file

raw_file = "raw_books_data.csv"

df_raw.to_csv(raw_file, index=False)

print(f"Raw dataset saved successfully as: {raw_file}")

Raw dataset saved successfully as: raw_books_data.csv


Download the raw dataset

In [ ]:
# Download the raw dataset to the local computer

from google.colab import files

files.download(raw_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>